# AML-GNN Thesis: Explainable Graph Neural Networks for Anti-Money Laundering

Code for the experimental component of the thesis *"Explainable Graph Neural Networks for Anti-Money Laundering: Evaluating Typology-Aligned Explanations in Transaction Monitoring."*

This codebase implements the 90 primary detection runs described in **Chapter 3, Section 3.11.1**:

| Factor | Options | Count |
| --- | --- | --- |
| Architectures | GCN, GAT, GraphSAGE | 3 |
| Imbalance strategies | Weighted CE, Focal Loss, GraphSMOTE | 3 |
| Datasets | Elliptic, IT-AML (HI-Small) | 2 |
| Random seeds | 0, 1, 2, 3, 4 | 5 |
| **Total** | | **90** |

---

## 1. Setup

### Python environment

In [ ]:
!unzip Archive.zip

Archive:  Archive.zip
   creating: src/
  inflating: __MACOSX/._src          
   creating: src/losses/
  inflating: __MACOSX/src/._losses   
  inflating: src/.DS_Store           
  inflating: __MACOSX/src/._.DS_Store  
   creating: src/training/
  inflating: __MACOSX/src/._training  
   creating: src/sampling/
  inflating: __MACOSX/src/._sampling  
 extracting: src/__init__.py         
  inflating: __MACOSX/src/.___init__.py  
   creating: src/models/
  inflating: __MACOSX/src/._models   
   creating: src/__pycache__/
  inflating: __MACOSX/src/.___pycache__  
  inflating: src/utils.py            
  inflating: __MACOSX/src/._utils.py  
   creating: src/data/
  inflating: __MACOSX/src/._data     
 extracting: src/losses/__init__.py  
  inflating: __MACOSX/src/losses/.___init__.py  
   creating: src/losses/__pycache__/
  inflating: __MACOSX/src/losses/.___pycache__  
  inflating: src/losses/losses.py    
  inflating: __MACOSX/src/losses/._losses.py  
  inflating: src/training/metrics.py  

In [ ]:
rm -rf /Users/wushaoxuan/PycharmProjects/aml_gnn_thesis/.venv

In [ ]:
%%bash
# Create a fresh virtual environment (recommended)

python -m venv .venv
source .venv/bin/activate   # macOS/Linux

# Install dependencies
pip install --upgrade pip
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.3 MB/s  0:00:00



Error: Command '['/content/.venv/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.
bash: line 4: .venv/bin/activate: No such file or directory


In [ ]:
pip install torch-geometric

> **PyTorch Geometric note.** If `pip install torch-geometric` fails on your system, install with the official wheel selector at https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html. PyG wheels are tied to your PyTorch and CUDA versions.

### Verify the installation

In [ ]:
import torch; import torch_geometric; print(torch.__version__, torch_geometric.__version__)

2.11.0+cu128 2.8.0


---

## 2. Datasets

### 2.1 Elliptic (auto-downloaded)

The Elliptic dataset is built into PyTorch Geometric and downloads automatically the first time it is loaded. Nothing to do manually.

In [ ]:
%%bash
# Optional: trigger the download now
python -m src.data.elliptic_loader

Elliptic dataset loaded successfully:
  num_nodes: 203769
  num_edges: 234355
  num_features: 165
  num_classes: 2
  train_count: 29894
  val_count: 5486
  test_count: 11184
  train_illicit: 3462
  val_illicit: 447
  test_illicit: 636
  train_time_steps: (1, 34)
  val_time_steps: (35, 39)
  test_time_steps: (40, 49)

Class distribution:
  train: n=29894, illicit=3462 (11.58%)
  val: n=5486, illicit=447 (8.15%)
  test: n=11184, illicit=636 (5.69%)


Processing...
Done!


The download goes to `data/elliptic/`.

### 2.2 IT-AML (manual download required)

The IBM IT-AML dataset must be downloaded from Kaggle:

1. Go to https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
2. Download the zip file (Kaggle account required; the dataset is free).
3. Extract into `data/it_aml/`, so the layout is:

In [ ]:
%%bash
mkdir -p data/it_aml

kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml

unzip -o ibm-transactions-for-anti-money-laundering-aml.zip -d ./data/it_aml/

ls -lh data/it_aml/

Dataset URL: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
License(s): Community Data License Agreement - Sharing - Version 1.0
Resuming from 2723151872 bytes (5453017546 bytes left)...

Archive:  ibm-transactions-for-anti-money-laundering-aml.zip
  inflating: ./data/it_aml/HI-Large_Patterns.txt  
  inflating: ./data/it_aml/HI-Large_Trans.csv  
  inflating: ./data/it_aml/HI-Large_accounts.csv  
  inflating: ./data/it_aml/HI-Medium_Patterns.txt  
  inflating: ./data/it_aml/HI-Medium_Trans.csv  
  inflating: ./data/it_aml/HI-Medium_accounts.csv  
  inflating: ./data/it_aml/HI-Small_Patterns.txt  
  inflating: ./data/it_aml/HI-Small_Trans.csv  
  inflating: ./data/it_aml/HI-Small_accounts.csv  
  inflating: ./data/it_aml/LI-Large_Patterns.txt  
  inflating: ./data/it_aml/LI-Large_Trans.csv  
  inflating: ./data/it_aml/LI-Large_accounts.csv  
  inflating: ./data/it_aml/LI-Medium_Patterns.txt  
  inflating: ./data/it_aml/LI-Medium_Trans.csv  
  i

100%|██████████| 7.61G/7.61G [04:48<00:00, 18.9MB/s]


In [ ]:
data/it_aml/
    HI-Small_Trans.csv
    HI-Small_Patterns.txt
    LI-Small_Trans.csv
    LI-Small_Patterns.txt
    ... (and other variants if downloaded)

SyntaxError: invalid syntax (1322901497.py, line 1)

Then verify:

In [ ]:
%%bash
python -m src.data.itaml_loader

[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
IT-AML Dataset loaded successfully:
  num_nodes: 515088
  num_edges: 5078345
  num_features: 10
  num_classes: 2
  train_count: 360599
  val_count: 77238
  test_count: 77251
  train_illicit: 5148
  val_illicit: 926
  test_illicit: 283
  has_typology_annotations: True

Typology distribution:
  None: 5,074,890
  GATHER-SCATTER: 723
  SCATTER-GATHER: 703
  STACK: 513
  CYCLE: 350
  FAN-OUT: 343
  FAN-IN: 323
  BIPARTITE: 267
  RANDOM: 233


> **If you don't have IT-AML yet**, you can still run the Elliptic experiments. See "Running the experiments" below.

---

## 3. Running the experiments

### 3.1 Smoke test (run 1 configuration, ~2-5 minutes)

In [ ]:
%%bash
python scripts/run_experiments.py --dataset elliptic --architecture gcn --imbalance_strategy weighted_ce --seed 0

09:33:18 | Project root: /content
09:33:18 | Will execute 1 configurations.
09:33:18 | First run: elliptic__gcn__weighted_ce__seed0
09:33:18 | Last run:  elliptic__gcn__weighted_ce__seed0
09:33:18 | 
=== Run 1/1 ===
09:33:18 | === elliptic__gcn__weighted_ce__seed0 | device=cuda ===
09:33:18 | Loading Elliptic dataset...
09:33:27 | Dataset info: {'num_nodes': 203769, 'num_edges': 234355, 'num_features': 165, 'num_classes': 2, 'train_count': 29894, 'val_count': 5486, 'test_count': 11184, 'train_illicit': 3462, 'val_illicit': 447, 'test_illicit': 636, 'train_time_steps': (1, 34), 'val_time_steps': (35, 39), 'test_time_steps': (40, 49)}
09:33:27 | Model: gcn | params=14,914
09:33:28 | Epoch   1 | loss=0.7795 | val_AUC-PR=0.1127 | val_F1=0.2111 | best_epoch=1
09:33:28 | Epoch  10 | loss=0.5835 | val_AUC-PR=0.2683 | val_F1=0.4080 | best_epoch=8
09:33:28 | Epoch  20 | loss=0.5086 | val_AUC-PR=0.2282 | val_F1=0.3773 | best_epoch=8
09:33:28 | Epoch  30 | loss=0.4952 | val_AUC-PR=0.2279 | val_F1

This runs a single GCN + Elliptic + Weighted CE + seed 0 configuration. Confirm you see:

- Dataset loaded successfully
- Training epochs printing every 10 epochs
- Final test metrics with AUC-PR > 0.1 (on Elliptic, a working baseline is around 0.4-0.6)
- A JSON result file in `results/elliptic__gcn__weighted_ce__seed0.json`

### 3.2 Elliptic only (30 runs)

In [ ]:
%%bash
python scripts/run_experiments.py --dataset elliptic

Process is terminated.


### 3.3 Full 90 runs (only after IT-AML is downloaded)

In [ ]:
%%bash
python scripts/run_experiments.py

[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,455 typology-labelled edges out of 5,078,345 total
[itaml_loader] Parsed 3,4

09:36:02 | Project root: /content
09:36:02 | Will execute 90 configurations.
09:36:02 | First run: elliptic__gcn__weighted_ce__seed0
09:36:02 | Last run:  itaml__graphsage__graphsmote__seed4
09:36:02 | 
=== Run 1/90 ===
09:36:02 | === elliptic__gcn__weighted_ce__seed0 | device=cuda ===
09:36:02 | Loading Elliptic dataset...
09:36:11 | Dataset info: {'num_nodes': 203769, 'num_edges': 234355, 'num_features': 165, 'num_classes': 2, 'train_count': 29894, 'val_count': 5486, 'test_count': 11184, 'train_illicit': 3462, 'val_illicit': 447, 'test_illicit': 636, 'train_time_steps': (1, 34), 'val_time_steps': (35, 39), 'test_time_steps': (40, 49)}
09:36:11 | Model: gcn | params=14,914
09:36:11 | Epoch   1 | loss=0.7795 | val_AUC-PR=0.1127 | val_F1=0.2111 | best_epoch=1
09:36:11 | Epoch  10 | loss=0.5835 | val_AUC-PR=0.2683 | val_F1=0.4080 | best_epoch=8
09:36:12 | Epoch  20 | loss=0.5086 | val_AUC-PR=0.2282 | val_F1=0.3773 | best_epoch=8
09:36:12 | Epoch  30 | loss=0.4952 | val_AUC-PR=0.2279 | va

### 3.4 Save results to Google Drive

After running the experiments, it's a good practice to save the `results/` and `logs/` directories to your Google Drive to ensure they are preserved and easily accessible.

In [ ]:
import os
import shutil
from datetime import datetime

# Define the destination path in Google Drive
drive_path = '/content/drive/MyDrive/AML_GNN_Thesis_Results'

# Create a timestamped folder to store results for this run
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
dest_folder = os.path.join(drive_path, f'experiment_results_{timestamp}')

# Ensure the destination folder exists in Drive
os.makedirs(dest_folder, exist_ok=True)

# Define source paths for results and logs
local_results_path = 'results/'
local_logs_path = 'logs/'

# --- Diagnostic additions start here ---
print(f"Checking for '{local_results_path}'...")
if os.path.exists(local_results_path):
    print(f"'{local_results_path}' found. Contents: {os.listdir(local_results_path)}")
else:
    print(f"'{local_results_path}' does not exist.")

print(f"Checking for '{local_logs_path}'...")
if os.path.exists(local_logs_path):
    print(f"'{local_logs_path}' found. Contents: {os.listdir(local_logs_path)}")
else:
    print(f"'{local_logs_path}' does not exist.")
# --- Diagnostic additions end here ---

# Copy the results directory to Google Drive
if os.path.exists(local_results_path):
    # Ensure the results destination exists before copying
    os.makedirs(os.path.join(dest_folder, 'results'), exist_ok=True)
    shutil.copytree(local_results_path, os.path.join(dest_folder, 'results'), dirs_exist_ok=True)
    print(f"Copied '{local_results_path}' to '{os.path.join(dest_folder, 'results')}'")
else:
    print(f"'{local_results_path}' not found. Skipping copy.")

# Optionally, copy the logs directory to Google Drive
if os.path.exists(local_logs_path):
    # Ensure the logs destination exists before copying
    os.makedirs(os.path.join(dest_folder, 'logs'), exist_ok=True)
    shutil.copytree(local_logs_path, os.path.join(dest_folder, 'logs'), dirs_exist_ok=True)
    print(f"Copied '{local_logs_path}' to '{os.path.join(dest_folder, 'logs')}'")
else:
    print(f"'{local_logs_path}' not found. Skipping copy.")

print(f"All experiment results and logs for this run are saved in your Google Drive at: {dest_folder}")

Checking for 'results/'...
'results/' found. Contents: ['_summary.json', 'elliptic__gcn__weighted_ce__seed0.json']
Checking for 'logs/'...
'logs/' found. Contents: ['run_experiments_20260607_141127.log', 'full_run_20260607_141118.log', 'run_experiments_20260608_093318.log']
Copied 'results/' to '/content/drive/MyDrive/AML_GNN_Thesis_Results/experiment_results_20260608_093424/results'
Copied 'logs/' to '/content/drive/MyDrive/AML_GNN_Thesis_Results/experiment_results_20260608_093424/logs'
All experiment results and logs for this run are saved in your Google Drive at: /content/drive/MyDrive/AML_GNN_Thesis_Results/experiment_results_20260608_093424


### 3.4 Aggregate results into a summary table

In [ ]:
%%bash
python scripts/aggregate_results.py

Aggregated summary written to: results/_summary_aggregated.csv
 dataset architecture imbalance_strategy  n_seeds  auc_pr_mean  auc_pr_std  auc_roc_mean  auc_roc_std  f1_at_threshold_mean  f1_at_threshold_std  recall_at_100_mean  recall_at_100_std  recall_at_200_mean  recall_at_200_std  recall_at_50_mean  recall_at_50_std
elliptic          gat              focal        5     0.306760    0.017974      0.831247     0.006763              0.343212             0.011958            0.095597           0.007267            0.178302           0.015489           0.049057          0.004266
elliptic          gat         graphsmote        5     0.289734    0.018423      0.820919     0.003099              0.334803             0.011138            0.089937           0.008344            0.155660           0.014238           0.052516          0.005031
elliptic          gat        weighted_ce        5     0.282752    0.017335      0.826281     0.002326              0.341131             0.007612            0

In [ ]:
import os

# Ensure the results directory exists before zipping
if os.path.exists('results/') and os.listdir('results/'):
    # Use the -r flag for recursive zipping and -q for quiet (optional)
    # The first argument is the name of the zip file to create
    # The second argument is the directory to zip
    !zip -r results.zip results/
    print("Successfully created results.zip")
else:
    print("The 'results/' directory is empty or does not exist. Skipping zipping.")

  adding: results/ (stored 0%)
  adding: results/elliptic__graphsage__graphsmote__seed2.json (deflated 61%)
  adding: results/elliptic__graphsage__focal__seed0.json (deflated 60%)
  adding: results/itaml__gcn__graphsmote__seed1.json (deflated 59%)
  adding: results/itaml__graphsage__graphsmote__seed3.json (deflated 60%)
  adding: results/elliptic__graphsage__focal__seed3.json (deflated 60%)
  adding: results/itaml__graphsage__focal__seed2.json (deflated 60%)
  adding: results/_summary.json (deflated 90%)
  adding: results/itaml__gcn__focal__seed2.json (deflated 59%)
  adding: results/itaml__gcn__weighted_ce__seed0.json (deflated 61%)
  adding: results/elliptic__gcn__graphsmote__seed0.json (deflated 60%)
  adding: results/elliptic__graphsage__focal__seed1.json (deflated 60%)
  adding: results/elliptic__gat__graphsmote__seed4.json (deflated 60%)
  adding: results/itaml__gcn__weighted_ce__seed2.json (deflated 59%)
  adding: results/elliptic__graphsage__weighted_ce__seed2.json (deflated 60